# Direct Gold Crypto Returns And Volatility 1d

Heavy notebook implementation for the Gold market crypto returns and realized volatility path.

This notebook reads cleaned Silver OHLC data, computes `simple_return_1d` and `log_return_1d`, derives trailing `7d`, `30d`, and `90d` realized volatility from daily returns, and merges the results into the two Gold market tables.

Execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed UTC day

The Silver and Gold target tables must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

Default product set: `BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD`.

In [ ]:
import uuid
from datetime import date, datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("product_ids", "BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "3")
dbutils.widgets.text("run_id", str(uuid.uuid4()))

CATALOG = dbutils.widgets.get("catalog").strip() or "market_macro"
RAW_PRODUCT_IDS = dbutils.widgets.get("product_ids").strip()
MODE = dbutils.widgets.get("mode").strip().lower()
START_DATE_RAW = dbutils.widgets.get("start_date").strip()
END_DATE_RAW = dbutils.widgets.get("end_date").strip()
LOOKBACK_DAYS_RAW = dbutils.widgets.get("lookback_days").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()

SOURCE_TABLE = f"{CATALOG}.slv_market.crypto_ohlc_1d"
RETURNS_TABLE = f"{CATALOG}.gld_market.dp_crypto_returns_1d"
VOLATILITY_TABLE = f"{CATALOG}.gld_market.dp_crypto_volatility_1d"
MAX_VOLATILITY_LOOKBACK_DAYS = 90
UTC = timezone.utc


In [ ]:
def parse_product_ids(raw_value: str) -> list[str]:
    product_ids = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    product_ids = list(dict.fromkeys(product_ids))
    if not product_ids:
        raise ValueError("product_ids cannot be empty")
    return product_ids


def parse_iso_date(field_name: str, raw_value: str) -> date:
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def resolve_date_window(
    mode: str,
    start_date_raw: str,
    end_date_raw: str,
    lookback_days_raw: str,
) -> tuple[date, date]:
    latest_complete_date = datetime.now(UTC).date() - timedelta(days=1)

    if mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")

        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        try:
            lookback_days = int(lookback_days_raw or "0")
        except ValueError as exc:
            raise ValueError("lookback_days must be an integer in incremental mode") from exc

        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed UTC day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def collect_product_counts(df, count_alias: str) -> dict[str, int]:
    counts = (
        df.groupBy("product_id")
        .count()
        .withColumnRenamed("count", count_alias)
        .collect()
    )
    return {row["product_id"]: int(row[count_alias]) for row in counts}


In [ ]:
def run_gold_crypto_returns_and_volatility() -> dict:
    product_ids = parse_product_ids(RAW_PRODUCT_IDS)
    start_date, end_date = resolve_date_window(MODE, START_DATE_RAW, END_DATE_RAW, LOOKBACK_DAYS_RAW)
    source_start_date = start_date - timedelta(days=MAX_VOLATILITY_LOOKBACK_DAYS)

    if not spark.catalog.tableExists(SOURCE_TABLE):
        raise RuntimeError(
            f"Source table {SOURCE_TABLE} does not exist. Run the Silver pipeline first."
        )

    if not spark.catalog.tableExists(RETURNS_TABLE):
        raise RuntimeError(
            f"Target table {RETURNS_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

    if not spark.catalog.tableExists(VOLATILITY_TABLE):
        raise RuntimeError(
            f"Target table {VOLATILITY_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

    silver_df = (
        spark.table(SOURCE_TABLE)
        .select(
            "product_id",
            "bar_date",
            "base_asset",
            "quote_currency",
            "close",
        )
        .filter(F.col("product_id").isin(product_ids))
        .filter((F.col("bar_date") >= F.lit(source_start_date)) & (F.col("bar_date") <= F.lit(end_date)))
    )

    rows_read = silver_df.count()
    per_product_rows_read = collect_product_counts(silver_df, "rows_read") if rows_read else {}

    if rows_read == 0:
        return {
            "status": "success_empty",
            "mode": MODE,
            "product_ids": product_ids,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_history_start_date": source_start_date.isoformat(),
            "source_table": SOURCE_TABLE,
            "returns_table": RETURNS_TABLE,
            "volatility_table": VOLATILITY_TABLE,
            "rows_read": 0,
            "rows_returns_ready": 0,
            "rows_volatility_ready": 0,
            "rows_returns_to_update": 0,
            "rows_returns_to_insert": 0,
            "rows_returns_merged": 0,
            "rows_volatility_to_update": 0,
            "rows_volatility_to_insert": 0,
            "rows_volatility_merged": 0,
            "run_id": RUN_ID,
            "per_product_rows_read": per_product_rows_read,
            "per_product_rows_returns": {},
            "per_product_rows_volatility": {},
        }

    order_window = Window.partitionBy("product_id").orderBy("bar_date")
    returns_base_df = (
        silver_df
        .withColumn("prev_close", F.lag("close").over(order_window))
        .withColumn(
            "simple_return_1d",
            F.when(
                F.col("prev_close") > 0,
                (F.col("close") / F.col("prev_close")) - F.lit(1.0),
            ),
        )
        .withColumn(
            "log_return_1d",
            F.when(
                (F.col("close") > 0) & (F.col("prev_close") > 0),
                F.log(F.col("close") / F.col("prev_close")),
            ),
        )
    )

    computed_at = datetime.now(UTC)
    returns_df = (
        returns_base_df
        .filter((F.col("bar_date") >= F.lit(start_date)) & (F.col("bar_date") <= F.lit(end_date)))
        .select(
            "product_id",
            "bar_date",
            "base_asset",
            "quote_currency",
            "close",
            "simple_return_1d",
            "log_return_1d",
        )
        .withColumn("computed_at", F.lit(computed_at))
        .withColumn("run_id", F.lit(RUN_ID))
    )

    rows_returns_ready = returns_df.count()
    per_product_rows_returns = collect_product_counts(returns_df, "rows_returns") if rows_returns_ready else {}

    window_7 = order_window.rowsBetween(-6, 0)
    window_30 = order_window.rowsBetween(-29, 0)
    window_90 = order_window.rowsBetween(-89, 0)

    volatility_base_df = (
        returns_base_df
        .withColumn("returns_count_7", F.count("simple_return_1d").over(window_7))
        .withColumn("returns_count_30", F.count("simple_return_1d").over(window_30))
        .withColumn("returns_count_90", F.count("simple_return_1d").over(window_90))
        .withColumn(
            "volatility_7d",
            F.when(F.col("returns_count_7") == 7, F.stddev_samp("simple_return_1d").over(window_7)),
        )
        .withColumn(
            "volatility_30d",
            F.when(F.col("returns_count_30") == 30, F.stddev_samp("simple_return_1d").over(window_30)),
        )
        .withColumn(
            "volatility_90d",
            F.when(F.col("returns_count_90") == 90, F.stddev_samp("simple_return_1d").over(window_90)),
        )
    )

    volatility_df = (
        volatility_base_df
        .filter((F.col("bar_date") >= F.lit(start_date)) & (F.col("bar_date") <= F.lit(end_date)))
        .select(
            "product_id",
            "bar_date",
            "base_asset",
            "quote_currency",
            "simple_return_1d",
            "volatility_7d",
            "volatility_30d",
            "volatility_90d",
        )
        .withColumn("computed_at", F.lit(computed_at))
        .withColumn("run_id", F.lit(RUN_ID))
    )

    rows_volatility_ready = volatility_df.count()
    per_product_rows_volatility = (
        collect_product_counts(volatility_df, "rows_volatility") if rows_volatility_ready else {}
    )

    existing_returns_key_count = (
        returns_df.select("product_id", "bar_date")
        .join(
            spark.table(RETURNS_TABLE).select("product_id", "bar_date"),
            on=["product_id", "bar_date"],
            how="inner",
        )
        .count()
    ) if rows_returns_ready else 0

    existing_volatility_key_count = (
        volatility_df.select("product_id", "bar_date")
        .join(
            spark.table(VOLATILITY_TABLE).select("product_id", "bar_date"),
            on=["product_id", "bar_date"],
            how="inner",
        )
        .count()
    ) if rows_volatility_ready else 0

    if rows_returns_ready > 0:
        DeltaTable.forName(spark, RETURNS_TABLE).alias("tgt").merge(
            returns_df.alias("src"),
            "tgt.product_id = src.product_id AND tgt.bar_date = src.bar_date",
        ).whenMatchedUpdate(
            set={
                "base_asset": "src.base_asset",
                "quote_currency": "src.quote_currency",
                "close": "src.close",
                "simple_return_1d": "src.simple_return_1d",
                "log_return_1d": "src.log_return_1d",
                "computed_at": "src.computed_at",
                "run_id": "src.run_id",
            }
        ).whenNotMatchedInsertAll().execute()

    if rows_volatility_ready > 0:
        DeltaTable.forName(spark, VOLATILITY_TABLE).alias("tgt").merge(
            volatility_df.alias("src"),
            "tgt.product_id = src.product_id AND tgt.bar_date = src.bar_date",
        ).whenMatchedUpdate(
            set={
                "base_asset": "src.base_asset",
                "quote_currency": "src.quote_currency",
                "simple_return_1d": "src.simple_return_1d",
                "volatility_7d": "src.volatility_7d",
                "volatility_30d": "src.volatility_30d",
                "volatility_90d": "src.volatility_90d",
                "computed_at": "src.computed_at",
                "run_id": "src.run_id",
            }
        ).whenNotMatchedInsertAll().execute()

    if rows_returns_ready > 0:
        display(returns_df.orderBy("product_id", "bar_date"))
    if rows_volatility_ready > 0:
        display(volatility_df.orderBy("product_id", "bar_date"))

    return {
        "status": "success",
        "mode": MODE,
        "product_ids": product_ids,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "source_history_start_date": source_start_date.isoformat(),
        "source_table": SOURCE_TABLE,
        "returns_table": RETURNS_TABLE,
        "volatility_table": VOLATILITY_TABLE,
        "rows_read": rows_read,
        "rows_returns_ready": rows_returns_ready,
        "rows_volatility_ready": rows_volatility_ready,
        "rows_returns_to_update": existing_returns_key_count,
        "rows_returns_to_insert": rows_returns_ready - existing_returns_key_count,
        "rows_returns_merged": rows_returns_ready,
        "rows_volatility_to_update": existing_volatility_key_count,
        "rows_volatility_to_insert": rows_volatility_ready - existing_volatility_key_count,
        "rows_volatility_merged": rows_volatility_ready,
        "run_id": RUN_ID,
        "per_product_rows_read": per_product_rows_read,
        "per_product_rows_returns": per_product_rows_returns,
        "per_product_rows_volatility": per_product_rows_volatility,
    }


result = run_gold_crypto_returns_and_volatility()
result
